# Lab Instructions

Create 3 visualizations from a spatial and time-series dataset of your choice.  Describe your dataset including where it came from and the features it contains.  Each visualization should be accompanied by at least 1 - 2 sentences explaining how the features do (or do not!) change over time and througout space.

In [8]:
import pandas as pd
import requests

# Set the date range for the data
start_date = '2024-01-01'
end_date = '2024-06-30'

# USGS API URL for all earthquakes with a magnitude >= 2.5
url = f'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}&minmagnitude=2.5'

# Fetch the data
response = requests.get(url)
data = response.json()

# Parse the GeoJSON data into a pandas DataFrame
features = data['features']
earthquakes = []
for feature in features:
    props = feature['properties']
    coords = feature['geometry']['coordinates']
    earthquakes.append({
        'time': pd.to_datetime(props['time'], unit='ms'),
        'latitude': coords[1],
        'longitude': coords[0],
        'depth': coords[2],
        'mag': props['mag'],
        'place': props['place']
    })

df = pd.DataFrame(earthquakes)

In [9]:
import plotly.express as px

fig = px.scatter_geo(df,
                     lat='latitude',
                     lon='longitude',
                     color='mag',
                     size='mag',
                     hover_name='place',
                     projection='natural earth',
                     title='Global Earthquake Epicenters (Magnitude >= 2.5)',
                     color_continuous_scale=px.colors.sequential.Plasma)
fig.show()

Global Earthquake Epicenters (Magnitude >= 2.5) - Scatter Map
This visualization shows the spatial distribution of earthquakes based on latitude and longitude. The size and color intensity correspond to the magnitude of the earthquake.
Explanation: The visualization reveals the locations of earthquakes worldwide, with higher-magnitude earthquakes shown by larger and more intense-colored markers. The data highlights that earthquakes occur primarily along the boundaries of tectonic plates, such as the Pacific Ring of Fire. While earthquakes are distributed globally, the clustering near plate boundaries indicates their relation to tectonic activity.

In [10]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Scatter(x=df['time'],
                                y=df['mag'],
                                mode='markers',
                                marker=dict(color=df['mag'],
                                            colorscale='Plasma',
                                            showscale=True,
                                            colorbar=dict(title='Magnitude')),
                                hovertext=df['place']))

fig.update_layout(title='Earthquake Magnitude Over Time',
                  xaxis_title='Date',
                  yaxis_title='Magnitude')
fig.show()

Earthquake Magnitude Over Time - Time Series Plot
This plot shows how earthquake magnitudes have varied over time, from January 1st to June 30th, 2024.
Explanation: The time series visualization reveals fluctuations in earthquake magnitude throughout the six-month period. Larger earthquakes are sporadically spaced, with no consistent pattern. The graph also allows you to observe periods of intense seismic activity, which could indicate geological phenomena, like plate movements or aftershocks from larger events.

In [11]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Scatter3d(x=df['longitude'],
                                   y=df['latitude'],
                                   z=df['depth'],
                                   mode='markers',
                                   marker=dict(
                                       size=5,
                                       color=df['mag'],
                                       colorscale='Viridis',
                                       opacity=0.8
                                   ))])

fig.update_layout(title='3D View of Global Earthquakes (Depth Included)',
                  scene=dict(
                      xaxis_title='Longitude',
                      yaxis_title='Latitude',
                      zaxis_title='Depth (km)'
                  ))
fig.show()

3D View of Global Earthquakes (Depth Included) - 3D Scatter Plot
This 3D scatter plot maps earthquakes based on their longitude, latitude, and depth, with color representing the earthquake's magnitude.
Explanation: This visualization shows the distribution of earthquakes not only in terms of location but also their depth. Deeper earthquakes are positioned further along the z-axis. From the plot, we see that most earthquakes occur at shallower depths, but there are also some deeper ones, especially around subduction zones, where one tectonic plate is sliding beneath another.